In [ ]:
# save as run_models_cv.py and run with: python run_models_cv.py
import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
from sklearn.impute import SimpleImputer
import importlib.util

CSV_PATH = "/mnt/data/92da2bec-b99c-4f8e-91ad-b02d9438d5be.csv"   # change if needed
OUT_DIR = "/mnt/data/model_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Load and inspect
df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH, "shape:", df.shape)
# detect target
possible_targets = ["defaulted", "default", "target", "y", "is_default"]
target_col = next((c for c in possible_targets if c in df.columns), df.columns[-1])
print("Using target column:", target_col)

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

# Basic preprocessing for non-numeric columns:
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
for c in non_numeric:
    nunique = X[c].nunique(dropna=True)
    if nunique <= 10:
        # one-hot small cardinality
        X = pd.get_dummies(X, columns=[c], prefix=[c], dummy_na=True)
    else:
        # frequency encode large cardinality
        freq = X[c].value_counts(normalize=True)
        X[c] = X[c].map(freq).fillna(0.0)

# Impute numeric missing values with median
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Models: logistic + XGBoost (fallback)
models = {}
models["LogisticRegression"] = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")

xgb_spec = importlib.util.find_spec("xgboost")
if xgb_spec is not None:
    import xgboost as xgb
    models["XGBoost"] = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss",
                                          random_state=42, n_estimators=200, max_depth=5, n_jobs=-1, verbosity=0)
    print("XGBoost found and will be used.")
else:
    models["GradientBoosting"] = GradientBoostingClassifier(random_state=42, n_estimators=200, max_depth=5)
    print("XGBoost not found; using sklearn GradientBoosting as fallback.")

# Cross-validation params
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

detailed_rows = []

for model_name, model in models.items():
    print("\nRunning CV for:", model_name)
    fold = 0
    for train_idx, test_idx in skf.split(X_imp, y):
        fold += 1
        X_tr, X_te = X_imp.iloc[train_idx], X_imp.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler().fit(X_tr)
        X_tr_s = scaler.transform(X_tr)
        X_te_s = scaler.transform(X_te)

        # Fit
        model.fit(X_tr_s, y_tr)

        # Predict probabilities
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_te_s)[:,1]
        else:
            try:
                scores = model.decision_function(X_te_s)
                y_proba = (scores - scores.min())/(scores.max()-scores.min()+1e-9)
            except Exception:
                y_proba = model.predict(X_te_s)

        y_pred = (y_proba >= 0.5).astype(int)

        # metrics
        auc = roc_auc_score(y_te, y_proba)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec = recall_score(y_te, y_pred, zero_division=0)
        acc = accuracy_score(y_te, y_pred)

        detailed_rows.append({
            "model": model_name, "fold": fold,
            "AUC": auc, "Precision": prec, "Recall": rec, "Accuracy": acc
        })
        print(f" {model_name} fold {fold}: AUC={auc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, Accuracy={acc:.4f}")

    # train final model on full data and save (with scaler)
    scaler_full = StandardScaler().fit(X_imp)
    model.fit(scaler_full.transform(X_imp), y)
    model_path = os.path.join(OUT_DIR, f"final_model_{model_name}.pkl")
    scaler_path = os.path.join(OUT_DIR, f"scaler_{model_name}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler_full, f)
    print(" Saved final model and scaler to:", model_path, scaler_path)

# Save detailed CV results and summary
detailed_df = pd.DataFrame(detailed_rows)
detailed_csv = os.path.join(OUT_DIR, "models_cv_detailed.csv")
detailed_df.to_csv(detailed_csv, index=False)
print("\nSaved detailed CV per-fold results to:", detailed_csv)

summary_rows = []
for model_name in detailed_df["model"].unique():
    s = detailed_df[detailed_df["model"]==model_name]
    summary_rows.append({
        "model": model_name,
        "AUC_mean": s["AUC"].mean(), "AUC_std": s["AUC"].std(),
        "Precision_mean": s["Precision"].mean(), "Precision_std": s["Precision"].std(),
        "Recall_mean": s["Recall"].mean(), "Recall_std": s["Recall"].std(),
        "Accuracy_mean": s["Accuracy"].mean(), "Accuracy_std": s["Accuracy"].std()
    })
summary_df = pd.DataFrame(summary_rows).round(4)
summary_csv = os.path.join(OUT_DIR, "models_cv_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print("Saved CV summary to:", summary_csv)
print("\nCV summary:\n", summary_df.to_string(index=False))
